#### The goal of this notebook is to build the dataset sample (from the whole dataset) that will be used for models training

In [18]:
import pandas as pd
import numpy as np
import random
import json
import tqdm

In [ ]:
# Open the dataset containing the reports and the paths to images
pd.options.display.max_columns = None
dataset = pd.read_csv("../CheXpert/df_chexpert_plus_240401.csv")
dataset = dataset.sort_values(by="path_to_image", ignore_index=True)
# "report" column contains the full raw doctor's report
#  Keeping columns with medical content: "section_findings", "section_impression", "section_summary" in case of future usage
columns_keep = ['path_to_image', 'deid_patient_id', 'report', 'section_findings', 'section_impression', 'section_summary']
dataset = dataset[columns_keep]
dataset

,path_to_image,deid_patient_id,report,section_findings,section_impression,section_summary
0,train/patient00001/study1/view1_frontal.jpg,patient00001,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\n \nLeft subclavian central venous catheter w...,\n \n1. Left subclavian central venous cathet...,NaN
1,train/patient00002/study1/view1_frontal.jpg,patient00002,NARRATIVE:\nFRONTAL AND LATERAL CHEST RADIOGRA...,\nThere is severe kyphosis centered at the mid...,\n1. MARKED KYPHOSIS WITH STATUS POST VERTEBRO...,"4: Possible Significant Abnormality/Change, m..."
2,train/patient00002/study1/view2_lateral.jpg,patient00002,NARRATIVE:\nFRONTAL AND LATERAL CHEST RADIOGRA...,\nThere is severe kyphosis centered at the mid...,\n1. MARKED KYPHOSIS WITH STATUS POST VERTEBRO...,"4: Possible Significant Abnormality/Change, m..."
3,train/patient00002/study2/view1_frontal.jpg,patient00002,"NARRATIVE:\nSINGLE VIEW OF THE CHEST, 4 VIEWS ...",NaN,\n1. SINGLE SUPINE PORTABLE FRONTAL VIEW OF TH...,"4-POSSIBLE SIGNIFICANT FINDINGS, MAY NEED ACTI..."
4,train/patient00003/study1/view1_frontal.jpg,patient00003,"NARRATIVE:\nCHEST, ONE VIEW: 2-10-2001\nFINDIN...","Costophrenic angles sharp, without evidence o...",\n1. NO EVIDENCE OF PNEUMOTHORAX.\n2. MILD INT...,2\nI have personally reviewed the images for ...
...,...,...,...,...,...,...
223457,valid/patient64736/study1/view1_frontal.jpg,patient64736,NARRATIVE:\nChest 1 View: 9/6/2003\n \nHISTORY...,NaN,\n \n1.THERE IS FREE INTRAPERITONEAL AIR SEEN...,"4-POSSIBLY SIGNIFICANT FINDING, MAY NEED ACTIO..."
223458,valid/patient64737/study1/view1_frontal.jpg,patient64737,NARRATIVE:\nPORTABLE CHEST RADIOGRAPH 1 VIEW: ...,NaN,\n1. INTERVAL REMOVAL OF RIGHT SUBCLAVIAN CENT...,"4-POSSIBLE SIGNIFICANT FINDINGS, MAY NEED ACTI..."
223459,valid/patient64738/study1/view1_frontal.jpg,patient64738,"NARRATIVE:\nChest 1 View, april 21\n \nHISTORY...",NaN,\n \n1.INTERVAL PLACEMENT OF A LEFT UPPER EX...,"2-ABNORMAL, PREVIOUSLY REPORTED\nI have perso..."
223460,valid/patient64739/study1/view1_frontal.jpg,patient64739,NARRATIVE:\nCOMPARISON: 7-21-2005\n \nIMPRESSI...,NaN,\n \nCARDIAC SILHOUETTE REMAINS MILDLY ENLARGE...,"2-ABNORMAL, PREVIOUSLY REPORTED\n \n"


#### To avoid data leakage, for each patient we keep one and only one x-ray image. The selection is random.

In [20]:
sub_dataset = dataset.groupby(by="deid_patient_id").sample(random_state=42)

# Pick randomly a subset of 20000 for the positive cases (the matches)
df_pos = sub_dataset.sample(20000, random_state=42)

#### One difficulty of the project is that we have to build the negative cases (the mismatches) manually, as the base dataset only contains images and their corresponding report

In [21]:
# Pick randomly a subset of 20000 for the nagative cases (the mismatches)
df_neg = sub_dataset.drop(index=df_pos.index).sample(20000, random_state=42)

# Dataframe where we look for mismatches
df_build_neg = dataset.drop(index=df_pos.index)

#### Building the mismatches is based on a swapping approach between reports that don't share at least one pathology

In [22]:
# Exploit impression_fixed.json file: for each report, pathologies are labelled
# 1: present, 0: absent, NaN: non mentioned, -1: uncertain 
impression = pd.read_json("../CheXpert/chexbert_labels/impression_fixed.json", lines=True)
impression = impression.sort_values(by="path_to_image", ignore_index=True)
# Fill NaN with numerical value 3
impression = impression.fillna(3)
# Adding the pathologies columns to dataframes df_neg and df_build_neg
df_neg = df_neg.merge(right=impression, on="path_to_image")
df_build_neg = df_build_neg.merge(right=impression, on="path_to_image")

In [ ]:
# Function that generates the mismatches
def generate_cluster_balanced_mismatch(df_neg, df_build_neg, total_samples=20000):
    df_neg = df_neg.reset_index(drop=True)
    df_build_neg = df_build_neg.reset_index(drop=True)
    
    pure_pathologies = [
        'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 
        'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 
        'Pleural Effusion', 'Pleural Other', 'Fracture'
    ]
    
    # Define clinical clusters mapping index positions to cluster groups
    # C1: Heart, C2: Opacities, C3: Pleural space, C4: Focal/Bone
    cluster_mapping = {
        0: 1, 1: 1,   # Enlarged Cardiomediastinum, Cardiomegaly
        2: 2, 4: 2, 5: 2, 6: 2, 7: 2,  # Lung Opacity, Edema, Consolidation, Pneumonia, Atelectasis
        8: 3, 9: 3, 10: 3, # Pneumothorax, Pleural Effusion, Pleural Other
        3: 4, 11: 4   # Lung Lesion, Fracture
    }
    
    matrix_neg = df_neg[pure_pathologies].values
    
    # Multi-dimensional sub-quotas
    quotas = {
        # Total (different cluster) 40%
        ("Total", "Diff"): int(total_samples * 0.4),
        # Solid (different cluster) 24%
        ("Solid", "Diff"): int(total_samples * 0.24),
        # Easy (different cluster) 16%
        ("Easy", "Diff"):  int(total_samples * 0.16),
        # Same clister whatever the type (Total or Solid or Easy) 20%
        ("All", "Same"): int(total_samples * 0.2)
    }
    
    counts = {k: 0 for k in quotas.keys()}
    mismatch_rows = []
    
    for i in range(len(matrix_neg)):
        if sum(counts.values()) >= total_samples:
            break
            
        matrix_build_neg = df_build_neg[pure_pathologies].values
        len_j = len(df_build_neg)
        vec_i = matrix_neg[i]
        pos_i = (vec_i == 1)
        
        start_j = random.randint(0, len_j - 1)
        
        for offset in range(len_j):
            j = (start_j + offset) % len_j
            
            vec_j = matrix_build_neg[j]
            pos_j = (vec_j == 1)
            
            total_i_to_j = np.all(vec_j[pos_i] == 0) if np.any(pos_i) else True
            total_j_to_i = np.all(vec_i[pos_j] == 0) if np.any(pos_j) else True
            is_total = total_i_to_j and total_j_to_i and (np.any(pos_i) or np.any(pos_j))
            has_diff = np.any((vec_i == 1) & (vec_j == 0)) or np.any((vec_j == 1) & (vec_i == 0))
            has_ambiguity = np.any((vec_i == 1) & np.isin(vec_j, [-1, 3])) or np.any((vec_j == 1) & np.isin(vec_i, [-1, 3]))
            
            tier_type = None
            if is_total:
                tier_type = "Total"
            elif has_diff and not has_ambiguity:
                tier_type = "Solid"
            elif has_diff and has_ambiguity:
                tier_type = "Easy"
                
            if not tier_type:
                continue  # Not a valid mismatch
                
            # --- Clustering Evaluation ---
            # Identify all pathologies present across BOTH individuals
            active_indices = np.where(((vec_i == 1) & (vec_j == 0)) | ((vec_i == 0) & (vec_j == 1)))[0]
            
            if len(active_indices) == 0:
                continue
                
            # Map those indices to their medical clusters
            active_clusters = set(cluster_mapping[idx] for idx in active_indices)
            
            # Determine if it's an intra-cluster (Same) or inter-cluster (Diff) mismatch
            cluster_type = "Same" if (len(active_clusters) == 1 and len(active_indices) != 1) else "Diff"
            if "Same" == cluster_type:
                tier_type = "All"
            
            # Composite Key lookup
            composite_key = (tier_type, cluster_type)
            
            # --- 3. Composite Quota Verification ---
            if counts[composite_key] < quotas[composite_key]:
                row_data = df_neg.iloc[i].copy()
                row_data["report"] = df_build_neg.loc[j, "report"]
                row_data["section_findings"] = df_build_neg.loc[j, "section_findings"]
                row_data["section_impression"] = df_build_neg.loc[j, "section_impression"]
                row_data["section_summary"] = df_build_neg.loc[j, "section_summary"]
                
                # Useful tracking tags for down-stream loss monitoring
                row_data["mismatch_tier"] = tier_type
                row_data["cluster_tier"] = cluster_type
                
                df_build_neg = df_build_neg.drop(index=j).reset_index(drop=True)
                mismatch_rows.append(row_data)
                counts[composite_key] += 1
                break 

    result_df = pd.DataFrame(mismatch_rows).reset_index(drop=True)
    print("Generation Successful! Sub-distribution counts:")
    for k, v in counts.items():
        print(f"  {k}: {v}")
    return result_df

In [ ]:
# Dataframe containing resulted mismatches
df_result_neg = generate_cluster_balanced_mismatch(df_neg, df_build_neg)

In [38]:
#Checking that we don't have a report from neither the matches dataset sample nor the initial mismatches dataset sample
for item in df_result_neg["report"]:
    if (item in df_neg["report"]) or (item in df_pos["report"]):
        print("Not good! Some reports are repeated")

#### Final step, building the dataset sample that contains 18000 matches and 18000 mismatches

In [39]:
SAMPLE_SIZE = 18000
df_result_neg = df_result_neg.sample(SAMPLE_SIZE, random_state=42)
df_result_neg["target"] = 0
df_pos = df_pos.sample(SAMPLE_SIZE, random_state=42)
df_pos["target"] = 1
dataset_sample = pd.concat([df_pos,df_result_neg], ignore_index=True)
dataset_sample = dataset_sample.sample(frac=1, random_state=42).reset_index(drop=True)
dataset_sample = dataset_sample.iloc[:, :7]

In [43]:
dataset_sample

,path_to_image,deid_patient_id,report,section_findings,section_impression,section_summary,target
0,train/patient29951/study1/view1_frontal.jpg,patient29951,NARRATIVE:\nChest 1 View: 7-8-2006\n \nHISTORY...,NaN,"\n \n1.LOW LUNG VOLUMES, WITH MINIMAL BASILAR ...",1-NO SIGNIFICANT ABNORMALITY \nI have personal...,1
1,train/patient03643/study1/view2_lateral.jpg,patient03643,NARRATIVE:\nCHEST AP PORTABLE: 11/1/2012\nCOMP...,NaN,\n1. REDEMONSTRATION OF A SMALL RIGHT PNEUMOTH...,"4: Possible significant abnormality/change, m...",0
2,train/patient32294/study1/view1_frontal.jpg,patient32294,NARRATIVE:\nTWO VIEWS OF THE CHEST: 3/1/2004.\...,NaN,\n \n1. PA AND LATERAL UPRIGHT VIEWS OF THE C...,"4-POSSIBLE SIGNIFICANT FINDINGS, MAY NEED ACTI...",0
3,train/patient33459/study1/view2_lateral.jpg,patient33459,NARRATIVE:\nEXAM: Chest 1 View december 23\n \...,NaN,\n \n1.SINGLE FRONTAL VIEW OF THE CHEST DEMONS...,"2-ABNORMAL, PREVIOUSLY REPORTED\nI have person...",0
4,train/patient01820/study2/view1_frontal.jpg,patient01820,"NARRATIVE:\nEXAM: Chest 2 Views, 12/20/16\n \n...",NaN,\n \n1.MILD RETICULAR OPACITIES ARE SEEN IN BO...,"4-POSSIBLY SIGNIFICANT FINDING, MAY NEED ACTIO...",1
...,...,...,...,...,...,...,...
35995,train/patient19113/study1/view1_frontal.jpg,patient19113,"NARRATIVE:\nEXAM: Chest 2 Views, 8-12-05.\n \n...",NaN,\n \n1.FRONTAL AND LATERAL RADIOGRAPHS OF THE ...,"2-ABNORMAL, PREVIOUSLY REPORTED\nI have person...",1
35996,train/patient48765/study5/view1_frontal.jpg,patient48765,NARRATIVE:\nSINGLE VIEW OF THE CHEST: 3/27/2...,Frontal radiograph of the chest performed o...,\n \n 1. INTERVAL REMOVAL OF LEFT CHEST TUB...,"4-POSSIBLE SIGNIFICANT FINDINGS, MAY NEED ACTI...",1
35997,train/patient45818/study3/view1_frontal.jpg,patient45818,"NARRATIVE:\nCHEST, ONE VIEW: 12/2/2013 1405\nC...",NaN,\n1. POSTOPERATIVE CHANGES WITH INTERVAL PLACE...,"2: ABNORMAL, PREVIOUSLY REPORTED\nI have pers...",1
35998,train/patient17330/study1/view1_frontal.jpg,patient17330,"NARRATIVE:\nPORTABLE CHEST, SINGLE VIEW: 9/9/2...",Single portable frontal view of the chest dem...,\n2. MODERATE INTERSTITIAL PULMONARY EDEMA WIT...,"4-POSSIBLE SIGNIFICANT FINDINGS, MAY NEED ACTI...",1


In [ ]:
# Save the dataset sample that will be used in models training into a csv file
dataset_sample.to_csv("../CheXpert/chexpert_plus_dataset_sample.csv")